# Code-Übung 2: Finite-Differenzen-Methode

## Stationäre Konvektions-Diffusions-Gleichung (2D-FDM)

Wir betrachten die Temperaturverteilung $\phi(x,y)$ in einem Fluid unter dem Einfluss einer Wärmequelle. Es wird angenommen, dass ein Gleichgewichtszustand vorherrscht, d.h. alle zeitlichen Ableitungen verschwinden. für diesen Fall gilt die stationäre Konvektions-Diffusions-Gleichung

$$
\rho c_p \mathbf{v} \cdot \nabla \phi(x,y) - \lambda \Delta \phi(x,y) = f(x,y)
$$

Hierbei beschreibt $\rho$ die Massendichte, $c_p$ die spezifische Wärmekapazität und $\lambda$ die Wärmeleitfähigkeit.
Im gesamten Gebiet sei eine laminare Strömung mit konstantem Vektor der Strömungsgeschwindigkeit $\mathbf{v}$ vorgegeben.
Die rechte Seite $f(x,y)$ beschreibt eine Wärmequelle. Diese soll in der Mitte des Gebiets wirken und ist hier examplarisch dargestellt. 

<p align=center><img src="./graphics/heat_source.png" style="width: 30%;"></p>

Außerdem sind Randbedingungen erforderlich. Hierfür wird eine Dirichlet-Randbedingung $\phi = \phi_0$ entlang des gesamten Randes verwendet.

## Numersiche Lösung

Das Gebiet wird mittels eines Gitters mit äquidistanten Schrittweiten diskretisiert. Die Ableitungen werden mit Zentraldifferenzen approximiert.
$$
K_E \Phi _{i-1, j} + K_S \Phi _{i, j-1} + K_P \Phi _ {i,j} + K_W \Phi _{i+1, j} + K_N \Phi _ {i, j+1} = f_{i,j}\\
K_{I,I-1} \Phi _{I-1} + K_{I,I-\bar{n}} \Phi _{I-\bar{n}} + K_{I, I} \Phi _ I + K _{I, I+1} \Phi _ {I+1} + K _ {I, I+\bar{n}} \Phi _{I+\bar{n}} = f _I\\
\textrm{wobei } \bar{n} = N_x + 2
$$
Die Koeffizienten $K$ bestehen dabei aus einem konvektiven und einem diffusiven Term.
$$
\textrm{Bsp.: } K_E = - \underbrace{\frac{\rho c_p v_1}{2 \Delta x}}_\text{konv.}  - \underbrace{\frac{\lambda}{\Delta x ^2}}_\text{diff.}
$$
Die geänderte Indexkonvention von $i, j$ zu $I$ kommt daher, dass für die korrekte Implementierung der Matrix die Umrechnung von den Gitterindizes $i, j$ zu den Indizes $I$ der Elemente vorgenommen wird (Bsp.: $(2, 3) \rightarrow 8$ für ein 5x5 Gitter). Die Nummerierung der Elemente startet dabei in der unteren linken Ecke der Platte. Die Umrechnung in Python lautet
$$
I = j \cdot N_x + i
$$
wobei zu beachten ist, dass in Python Indizes ab $0$ gezählt werden (Das Bsp. von oben lautet dann $(2, 3) \rightarrow 13$, da $i_{Py}=0$, $j_{Py}=0$ und $I_{Py}=0$ ebenfalls dazugezählt werden).\
Wie bereits bei den vorherigen 1D-Beispielen, liegen beim Anwenden der Zentraldifferenz einige Punkte außerhalb des Gitters. Hierfür werden wieder Dirichlet-Randbedingungen genutzt, die analog zu den anderen Beipsielen implementiert sind. Der Wärmestrom in der Mitte der Platte geht über den Lastvektor in die Rechnung ein.

### Import

In [ ]:
import sys
import numpy as np
from scipy.linalg import solve

# import helper functions, depending on whether we are running in Google Colab or not

IN_COLAB = "google.colab" in sys.modules # check if we are running in Google Colab (if module exists, we are)

if IN_COLAB:
    # clone the repository to access the helper functions (the version on the main branch)
    !git clone https://github.com/CPShub/LectureNSM.git 
    from LectureNSM.ex2_finite_differences.helper_functions import plot_heat
    %rm -rf LectureNSM  # delete the folder after importing, to enable re-importing if we run the cell again

else:
    from helper_functions import plot_heat

### Modellparameter

In [ ]:
# Model parameters
L_x = 1.0                   # x dimension of the plate in m
L_y = 1.0                   # y dimension of the plate in m
rho = 1                 # Density of the medium in kg/m^3 (1.0e3 for water)
c_p  = 1               # heat capacity in J/(kg K) (4.18e3 for water)
lambd   = 1               # thermal conductivity in W/(m K) (0.6 for water)
alpha = lambd/(rho*c_p)     # thermal diffusivity in m^2/s
v_1 = 0.0                     # x speed in m/s
v_2 = 0.0                     # y speed in m/s

# Grid parameters
N_x = 11                    # Discrete points in x direction
N_y = 11                    # Discrete points in y direction
delta_x = L_x/(N_x-1)       # Step size in x direction in m
delta_y = L_y/(N_y-1)       # Step size in y direction in m
N = N_x * N_y               # Total degrees of freedom
K = np.zeros((N, N))        # Stiffness matrix
f = np.zeros((N+2, 1))      # Load vector
node_transform = lambda i, j: j*N_x + i  # Conversion from grid index (i, j) to degree of freedom index (I) ( = row in load vector)

### Aufstellen des Gleichungssystems für innere Knoten

In [ ]:
# Coefficients of inner nodes
a_W = lambda idx: -(rho*c_p*v_1)/(2*delta_x) - lambd/(delta_x**2)
a_E = lambda idx: (rho*c_p*v_1)/(2*delta_x) - lambd/(delta_x**2)
a_P = lambda idx: (2*lambd)/(delta_x**2)+(2*lambd)/(delta_y**2)
a_N = lambda idx: (rho*c_p*v_2)/(2*delta_y) - lambd/(delta_y**2)
a_S = lambda idx: -(rho*c_p*v_2)/(2*delta_y) - lambd/(delta_y**2)

# Assembling the stiffness matrix for all the inner nodes
for i in range(1, N_x-1):
    for j in range(1, N_y-1):            # Note: Border elements are ommited for now
        I = node_transform(i, j)                

        K[I][I-1] = a_W(I)               # Note: Different sign as previously examples due to new definition
        K[I][I-N_x] = a_S(I)
        K[I][I] = a_P(I)
        K[I][I+1] = a_E(I)
        K[I][I+N_x] = a_N(I)

# Assemble the load vector (here: Heat source)
# Define heat source in spatial coordinates first (dimension N_x x N_y)
source_size = 3                        # dimension of source in number of points per side
Q_0 = 1000                             # heat flux in W
heat_source = np.zeros((N_x, N_y))
heat_source[round(N_x/2-source_size/2):round(N_x/2+source_size/2), round(N_y/2-source_size/2):round(N_y/2+source_size/2)] = Q_0

# Flattening the heat source to be a vector (each row now corresponds to one element)
f = heat_source.flatten()

### Einbinden der Randbedingungen

In [ ]:
# Add Dirichlet boundary conditions at the edge of the plate. This adds info to rows that were previously skipped when only
# looking at inner nodes
T_0 = 300                               # Temperature at the edge in K

for i in range(N_x):                    # Top and bottom edge
  for j in [0, N_y-1]:
    I = node_transform(i, j) 
    K[I,I] = 1
    f[I] = T_0

for j in range(1, N_y-1):               # Left and right edge                          
  for i in [0, N_x-1]:
    I = node_transform(i, j) 
    K[I,I] = 1
    f[I] = T_0

### Lösen des LGS

In [ ]:
phi = solve(K, f)

### Visualisieren der Ergebnisse

In [ ]:
# Plotting the results
x = np.linspace(0, L_x, N_x)
y = np.linspace(0, L_y, N_y)
X, Y = np.meshgrid(x, y)
Phi = phi.reshape(N_y, N_x)
plot_heat(heat_source, X, Y, Phi)

### Interpretation der Ergebnisse

Je nachdem wie die Modellparameter gewählt werden kann das Verfahren zu unphysikalischen Ergebnissen führen, welche dadurch zustande kommen, dass das Verfahren instabil ist.

* Für reine Diffusion, also $v_x = v_y = 0$ ist das Verfahren immer stabil.
* Liegt Konvektion vor, so ist das Verfahren nur bereichsweise stabil.

Die Grenze ab der Instabilitäten auftreten wird durch eine Bedingung für die Peclet-Zahl $\mathrm{Pe}_h$ mit allgemeiner Gitterweite $h$ beschrieben.
In diesem Falle muss gelten $\mathrm{Pe}_h~=~\dfrac{c_p\rho v h}{\alpha} \leq 2 $. Für die Wahl der Gitterweiten folgt also:
$$
\Delta x \leq \frac{2\alpha}{\rho c_p v_x} \;\;\; \mathrm{und} \;\;\; \Delta y \leq \frac{2\alpha}{\rho c_p v_y}
$$
Hierauf wird im Kontext der FVM näher eingegangen.
